<a href="https://colab.research.google.com/github/MufazaMajeed09/ai-recommendation-ranking-platform/blob/main/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AI-Powered Personalized Learning Recommendation & Ranking Platform**

Goal: Recommend and rank the most relevant learning resources for a student based on their skills, interests, previous performance, learning history, and goals.


In [1]:
print("AI-Powered Learning Recommendation & Ranking Platform")

AI-Powered Learning Recommendation & Ranking Platform


## Diagnosing the OULAD zip extraction error

Symptom: `zipfile.is_zipfile(...)` returns `True` and `namelist()` works, but `ZipFile.extract()` raises `OSError: [Errno 22] Invalid argument` — even for a small file like `courses.csv`.

The zip lives at `/content/open+university+learning+analytics+dataset.zip` (local Colab disk, not a Google Drive mount), so this cell gathers evidence rather than assuming a cause. It is **read-only with respect to the original zip** (opened in mode `"r"` only) and only ever writes to a throwaway scratch path that it deletes afterward. It checks, in order:

1. Where the zip actually is and how much free disk space `/content` has.
2. Whether the archive's data is intact (`testzip()` — a full CRC check).
3. The raw `ZipInfo` metadata (compression, flags, ZIP64 usage) for the file that fails.
4. Whether the failure is on the **read** side (`zf.open().read()`, fully in-memory, no disk write) or the **write** side (a plain manual `open(path, "wb").write()`, bypassing `zipfile`'s own extraction code path).
5. A reproduction of the original `extract()` failure with the full traceback, so we can see exactly which internal operation raises the error.

No extraction fix is applied yet — this only diagnoses.

In [ ]:
import zipfile
import os
import shutil
import traceback

ZIP_PATH = "/content/open+university+learning+analytics+dataset.zip"
TARGET_ENTRY = "courses.csv"
SCRATCH_DIR = "/content/_diag_scratch_dir"
SCRATCH_FILE = "/content/_diag_scratch_courses.csv"

# 1. Confirm where the zip really is and how much free disk space /content has
print("=== Location & disk space ===")
print("Zip exists:", os.path.exists(ZIP_PATH))
print("Zip size (bytes):", os.path.getsize(ZIP_PATH))
resolved = os.path.realpath(ZIP_PATH)
print("Resolved path:", resolved)
print("Is under a Google Drive mount:", resolved.startswith("/content/drive"))

total, used, free = shutil.disk_usage("/content")
print(f"/content disk space -> total: {total/1e9:.2f} GB, used: {used/1e9:.2f} GB, free: {free/1e9:.2f} GB")

# Everything below opens the original zip strictly read-only ("r") -- it is never
# written to, moved, or deleted.
with zipfile.ZipFile(ZIP_PATH, mode="r") as zf:
    names = zf.namelist()
    print(f"\n=== Archive contents ===")
    print(f"Entries in archive: {len(names)}")
    print("First 10 entries:", names[:10])

    # 2. Integrity check -- reads every entry's compressed data and verifies its CRC.
    # Still read-only; does not extract or write anything to disk.
    print("\n=== Integrity check (testzip) ===")
    bad_file = zf.testzip()
    print("First corrupted entry (None = no corruption found):", bad_file)

    # 3. Raw metadata for the entry that fails to extract
    print(f"\n=== ZipInfo for {TARGET_ENTRY!r} ===")
    info = zf.getinfo(TARGET_ENTRY)
    print("  compress_type:", info.compress_type, "(0=stored, 8=deflated)")
    print("  create_system:", info.create_system, "(0=Windows/FAT, 3=Unix)")
    print("  create_version:", info.create_version)
    print("  extract_version:", info.extract_version)
    print("  flag_bits:", bin(info.flag_bits))
    print("  header_offset:", info.header_offset)
    print("  compress_size:", info.compress_size)
    print("  file_size:", info.file_size)
    print("  date_time:", info.date_time)
    requires_zip64 = info.file_size > zipfile.ZIP64_LIMIT or info.compress_size > zipfile.ZIP64_LIMIT
    print("  Requires ZIP64:", requires_zip64)

    # 4. Isolate READ vs WRITE: read the entry fully into memory via zf.open().
    # This exercises the decompression path but performs no filesystem write at all.
    print(f"\n=== In-memory read via zf.open() ===")
    read_ok = False
    data = None
    try:
        with zf.open(TARGET_ENTRY) as f:
            data = f.read()
        print(f"In-memory read succeeded: {len(data)} bytes")
        read_ok = True
    except Exception:
        print("In-memory read FAILED:")
        traceback.print_exc()

    # 5. If the in-memory read worked, try a plain manual write, which bypasses
    # zipfile's own extract() write path (whose internals we're trying to isolate).
    if read_ok:
        print(f"\n=== Manual write (bypassing zf.extract()) ===")
        try:
            with open(SCRATCH_FILE, "wb") as out:
                out.write(data)
            print(f"Manual write to {SCRATCH_FILE} succeeded.")
        except Exception:
            print("Manual write FAILED:")
            traceback.print_exc()
        finally:
            if os.path.exists(SCRATCH_FILE):
                os.remove(SCRATCH_FILE)
                print("Scratch file removed.")

    # 6. Reproduce the original extract() failure with a full traceback, so we can
    # see exactly which internal call raises OSError 22.
    print(f"\n=== Reproducing zf.extract({TARGET_ENTRY!r}) ===")
    try:
        zf.extract(TARGET_ENTRY, path=SCRATCH_DIR)
        print("extract() unexpectedly succeeded this time.")
    except Exception:
        print("extract() FAILED (as originally reported):")
        traceback.print_exc()
    finally:
        shutil.rmtree(SCRATCH_DIR, ignore_errors=True)

print("\nDiagnostic complete. No changes were made to the original zip file.")

## Extracting the 7 OULAD CSVs

Safe extraction of all `.csv` members from the zip into a separate output folder on local Colab disk. The original zip is opened read-only and is never modified, moved, or deleted.

It tries `zf.extractall()` first. If that hits the same `OSError: [Errno 22]`, it falls back automatically to extracting file-by-file via `zf.open().read()` + a plain manual write, which sidesteps whichever internal `zipfile.extract()` operation was raising the error.

In [ ]:
import zipfile
import os

ZIP_PATH = "/content/open+university+learning+analytics+dataset.zip"
OUT_DIR = "/content/oulad_extracted"

os.makedirs(OUT_DIR, exist_ok=True)

# Opened strictly read-only ("r") -- the original zip is never modified, moved, or deleted.
with zipfile.ZipFile(ZIP_PATH, mode="r") as zf:
    csv_members = [n for n in zf.namelist() if n.lower().endswith(".csv")]
    print(f"Found {len(csv_members)} CSV member(s) in the archive:")
    for name in csv_members:
        print(" -", name)

    try:
        # Fast path: works when the underlying OS/filesystem issue isn't present.
        zf.extractall(path=OUT_DIR, members=csv_members)
        print(f"\nextractall() succeeded -- all CSVs written to {OUT_DIR}")
    except OSError as e:
        # Fallback: read each entry fully into memory, then write it out with a
        # plain file handle. This bypasses whichever internal zipfile.extract()
        # operation was raising OSError 22 in the diagnostic cell above.
        print(f"\nextractall() failed ({e}); falling back to manual per-file extraction...")
        for name in csv_members:
            dest_path = os.path.join(OUT_DIR, os.path.basename(name))
            with zf.open(name) as src, open(dest_path, "wb") as dst:
                dst.write(src.read())
            print(f"  extracted {name} -> {dest_path}")

# Verify: list what actually landed on disk and confirm sizes line up.
print(f"\n=== Contents of {OUT_DIR} ===")
extracted = sorted(os.listdir(OUT_DIR))
for fname in extracted:
    fpath = os.path.join(OUT_DIR, fname)
    print(f"  {fname}  ({os.path.getsize(fpath):,} bytes)")

print(f"\nExtracted {len(extracted)} of {len(csv_members)} expected CSV files.")
assert len(extracted) == len(csv_members), "Mismatch: not all CSVs were extracted."